In [1]:
import pandas as pd
import numpy as np

In [2]:
fi = pd.read_csv("Data/Context/Finland.csv")

In [3]:
fi = fi.drop(["Unnamed: 0"], axis=1)

In [4]:
fi_2007_2013 = fi.loc[7:13]

In [5]:
fi_2007_2013

,TIME_PERIOD,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,...,WTDA,EDBS,DBP,DBD,SSB,DWCP,PMSI,PT,EC,ELFE
7,2007,1.90,1.97,2.39,1.55,1.46,94.0,0.985,0.971,80.8,...,5.38,NaN,3.0,17.0,91.081035,76.259876,56.666667,68.824462,78.366876,NaN
8,2008,1.91,2.04,2.32,1.61,1.46,90.0,0.985,0.971,83.7,...,5.46,NaN,3.0,17.0,92.303567,76.532869,56.666667,68.783401,78.383373,NaN
9,2009,1.96,2.23,2.24,1.81,1.47,89.0,0.985,0.971,82.5,...,5.36,NaN,3.0,17.0,92.332122,76.560705,56.666667,68.783401,78.366876,NaN
10,2010,1.96,2.22,2.15,1.87,1.49,92.0,0.985,0.971,86.9,...,5.15,80.202957,3.0,17.0,92.345580,77.065080,56.666667,76.827451,74.541739,NaN
11,2011,1.94,2.24,2.19,1.81,1.51,94.0,0.985,0.971,88.7,...,5.23,80.641955,3.0,17.0,92.292541,76.916557,56.666667,80.247713,74.541739,NaN
12,2012,1.94,2.21,2.23,1.82,1.60,90.0,0.985,0.971,89.9,...,5.45,81.393609,3.0,17.0,92.333827,77.022476,56.666667,88.839006,73.454375,NaN
13,2013,1.96,2.18,2.20,1.84,1.57,89.0,0.985,0.981,91.5,...,5.74,81.402915,3.0,17.0,92.363760,77.112896,56.666667,88.088291,73.454375,NaN


In [6]:
def missing_values(df):
    for feature in df.columns:
        missing_values = df[feature].isna().sum()
        percentage = missing_values/len(df) * 100
        print(f'{feature} - {missing_values} - {round(percentage, 1)} %')

In [7]:
missing_values(fi_2007_2013)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
VAA - 0 - 0.0 %
CPI - 0 - 0.0 %
FOP - 0 - 0.0 %
JI - 0 - 0.0 %
IUI - 0 - 0.0 %
CFI - 4 - 57.1 %
PR - 0 - 0.0 %
DI - 2 - 28.6 %
SOARS - 0 - 0.0 %
TOGP - 0 - 0.0 %
IPAB - 3 - 42.9 %
PTIP - 0 - 0.0 %
QOES - 0 - 0.0 %
AOLT - 0 - 0.0 %
FLTA - 0 - 0.0 %
EBOF - 0 - 0.0 %
FOWD - 0 - 0.0 %
QOMS - 0 - 0.0 %
CLER - 0 - 0.0 %
WTDA - 0 - 0.0 %
EDBS - 3 - 42.9 %
DBP - 0 - 0.0 %
DBD - 0 - 0.0 %
SSB - 0 - 0.0 %
DWCP - 0 - 0.0 %
PMSI - 0 - 0.0 %
PT - 0 - 0.0 %
EC - 0 - 0.0 %
ELFE - 7 - 100.0 %


In [8]:
mode = fi_2007_2013['DI'].mode()[0]
fi_2007_2013['DI'].fillna(value=mode, inplace=True)

/var/folders/41/w_0mz3gn2vn0w0dz8wywr8_r0000gn/T/ipykernel_34499/1913970909.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fi_2007_2013['DI'].fillna(value=mode, inplace=True)


In [9]:
fi_2007_2013 = fi_2007_2013.drop(["ELFE", "CFI", "IPAB", "EDBS"], axis=1)

In [10]:
def fill_with_mean(df):
    for feature in df.columns:
        mean = df[feature].mean()
        df[feature].fillna(value=mean, inplace=True)

In [11]:
continuous_data = ['DBP', "DBD", "EDBS", "SSB", "DWCP", "PT", "EC", "IUI", "PMSI"]

In [12]:
def handle_outliers(df, continuous_features):

    print("Original Shape:", df.shape)
    
    outlier_summary = {}
    
    for feature in continuous_features:
        if feature in df.columns:
            Q1 = df[feature].quantile(0.25)
            Q3 = df[feature].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outlier_mask = (df[feature] < lower_bound) | (df[feature] > upper_bound)
            num_outliers = outlier_mask.sum()
            
            if num_outliers > 0:
                non_outlier_mean = df.loc[~outlier_mask, feature].mean()
                
                df.loc[outlier_mask, feature] = non_outlier_mean
                
                print(f"{feature}: Found {num_outliers} outliers, replaced with mean = {non_outlier_mean:.2f}")
            else:
                print(f"{feature}: No outliers found")
    
    print("\nFinal Shape:", df.shape)
    print(f"Total features processed: {len(continuous_features)}")
    
    return df

In [13]:
fi_2007_2013_no_o = handle_outliers(fi_2007_2013, continuous_data)

Original Shape: (7, 30)
DBP: No outliers found
DBD: No outliers found
SSB: Found 1 outliers, replaced with mean = 92.33
DWCP: No outliers found
PT: No outliers found
EC: No outliers found
IUI: No outliers found
PMSI: No outliers found

Final Shape: (7, 30)
Total features processed: 9


In [14]:
fi_2007_2013_no_o

,TIME_PERIOD,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,...,QOMS,CLER,WTDA,DBP,DBD,SSB,DWCP,PMSI,PT,EC
7,2007,1.90,1.97,2.39,1.55,1.46,94.0,0.985,0.971,80.8,...,5.53,5.14,5.38,3.0,17.0,92.328566,76.259876,56.666667,68.824462,78.366876
8,2008,1.91,2.04,2.32,1.61,1.46,90.0,0.985,0.971,83.7,...,5.53,5.32,5.46,3.0,17.0,92.303567,76.532869,56.666667,68.783401,78.383373
9,2009,1.96,2.23,2.24,1.81,1.47,89.0,0.985,0.971,82.5,...,5.44,5.26,5.36,3.0,17.0,92.332122,76.560705,56.666667,68.783401,78.366876
10,2010,1.96,2.22,2.15,1.87,1.49,92.0,0.985,0.971,86.9,...,5.28,5.33,5.15,3.0,17.0,92.345580,77.065080,56.666667,76.827451,74.541739
11,2011,1.94,2.24,2.19,1.81,1.51,94.0,0.985,0.971,88.7,...,5.33,5.24,5.23,3.0,17.0,92.292541,76.916557,56.666667,80.247713,74.541739
12,2012,1.94,2.21,2.23,1.82,1.60,90.0,0.985,0.971,89.9,...,5.55,5.22,5.45,3.0,17.0,92.333827,77.022476,56.666667,88.839006,73.454375
13,2013,1.96,2.18,2.20,1.84,1.57,89.0,0.985,0.981,91.5,...,5.64,5.15,5.74,3.0,17.0,92.363760,77.112896,56.666667,88.088291,73.454375


In [15]:
fi_2007_2013_no_o.to_csv('Data/Context/Finland_2007_2013.csv')

In [16]:
fi_2007_2013_no_o = fi_2007_2013_no_o.drop(["TIME_PERIOD"], axis=1)

In [17]:
fi_2007_2013_l = fi_2007_2013_no_o.loc[13]

In [18]:
fi_2007_2013_l.to_csv('Data/Context/Finland_2007_2013_last.csv')

In [19]:
fi_2014_2020 = fi.loc[14:20]

In [20]:

missing_values(fi_2014_2020)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
VAA - 0 - 0.0 %
CPI - 0 - 0.0 %
FOP - 0 - 0.0 %
JI - 0 - 0.0 %
IUI - 0 - 0.0 %
CFI - 0 - 0.0 %
PR - 0 - 0.0 %
DI - 0 - 0.0 %
SOARS - 3 - 42.9 %
TOGP - 3 - 42.9 %
IPAB - 3 - 42.9 %
PTIP - 3 - 42.9 %
QOES - 3 - 42.9 %
AOLT - 3 - 42.9 %
FLTA - 3 - 42.9 %
EBOF - 3 - 42.9 %
FOWD - 3 - 42.9 %
QOMS - 3 - 42.9 %
CLER - 3 - 42.9 %
WTDA - 4 - 57.1 %
EDBS - 0 - 0.0 %
DBP - 0 - 0.0 %
DBD - 0 - 0.0 %
SSB - 0 - 0.0 %
DWCP - 0 - 0.0 %
PMSI - 0 - 0.0 %
PT - 0 - 0.0 %
EC - 0 - 0.0 %
ELFE - 1 - 14.3 %


In [21]:
fi_2014_2020 = fi_2014_2020.drop(["SOARS", "TOGP", "IPAB", "PTIP", "QOES", "AOLT", "FLTA", "EBOF",
                                 "FOWD", "QOMS", "CLER", "WTDA"], axis=1)

In [22]:
fill_with_mean(fi_2014_2020)

In [23]:
continuous_data = ['DBP', "DBD", "EDBS", "SSB", "DWCP", "PT", "EC", "IUI", "PMSI"]

In [24]:
fi_2014_2020_no_o = handle_outliers(fi_2014_2020, continuous_data)

Original Shape: (7, 22)
DBP: No outliers found
DBD: Found 1 outliers, replaced with mean = 17.00
EDBS: Found 1 outliers, replaced with mean = 79.98
SSB: Found 1 outliers, replaced with mean = 92.39
DWCP: Found 1 outliers, replaced with mean = 73.42
PT: No outliers found
EC: Found 1 outliers, replaced with mean = 66.40
IUI: No outliers found
PMSI: No outliers found

Final Shape: (7, 22)
Total features processed: 9


In [25]:
fi_2014_2020_no_o.to_csv('Data/Context/Finland_2014_2020.csv')

In [26]:
fi_2014_2020_no_o = fi_2014_2020_no_o.drop(["TIME_PERIOD"], axis=1)

In [27]:
fi_2014_2020_no_o

,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,CFI,...,DI,EDBS,DBP,DBD,SSB,DWCP,PMSI,PT,EC,ELFE
14,2.12,2.03,2.16,1.88,1.54,89.0,0.985,0.981,86.5,60.7,...,9.03,80.200594,3.0,17.0,92.362866,73.421691,62.0,88.454135,66.397984,8.183333
15,2.05,1.84,2.22,1.83,1.54,90.0,0.969,0.978,86.4,60.0,...,9.03,79.978237,3.0,17.0,92.360986,73.423990,62.0,88.355975,66.397984,9.200000
16,2.01,1.84,2.19,1.81,1.53,89.0,0.969,0.978,87.7,59.9,...,9.03,79.867021,3.0,17.0,92.372939,73.446840,62.0,90.309544,66.397984,7.900000
17,2.02,1.98,2.16,1.81,1.56,85.0,0.977,0.978,87.5,58.5,...,9.03,79.860023,3.0,17.0,92.393599,73.340151,62.0,90.234946,66.397984,8.200000
18,2.03,2.01,2.17,1.78,1.59,85.0,0.984,0.983,88.9,59.6,...,9.14,79.746609,3.0,17.0,92.410229,73.359369,62.0,90.135735,66.397984,8.300000
19,2.01,1.97,2.11,1.84,1.56,86.0,0.979,0.978,89.6,59.8,...,9.25,79.803902,3.0,17.0,92.431874,73.407576,62.0,90.641569,66.397984,7.300000
20,2.02,1.90,2.17,1.85,1.61,85.0,0.983,0.981,92.2,57.0,...,9.20,80.391274,3.0,17.0,92.388749,73.552220,62.0,90.897460,66.397984,8.200000


In [28]:
fi_2014_2020_l = fi_2014_2020_no_o.loc[20]

In [29]:
fi_2014_2020_l

ROL      2.020000
GE       1.900000
COC      2.170000
RQ       1.850000
VAA      1.610000
CPI     85.000000
FOP      0.983000
JI       0.981000
IUI     92.200000
CFI     57.000000
PR      92.000000
DI       9.200000
EDBS    80.391274
DBP      3.000000
DBD     17.000000
SSB     92.388749
DWCP    73.552220
PMSI    62.000000
PT      90.897460
EC      66.397984
ELFE     8.200000
Name: 20, dtype: float64

In [30]:
fi_2014_2020_l.to_csv('Data/Context/Finland_2014_2020_last.csv')